# 从零实现 Double DQN：Q 网络、经验回放与目标网络

本 Notebook 用 PyTorch 手写 `QNetwork.forward`，并搭建一个完整但很小的强化学习闭环：环境交互、epsilon-greedy、replay buffer、Double DQN target、Huber loss、target network 同步、离线评估和可信策略制品。

重点不是“跑出一个高 reward”，而是验证 terminal mask、target detach、online/target 网络职责、随机性和服务边界。受控链式环境的成功不能外推 Atari、机器人或线上推荐。

## 1. 可复现运行合同

环境、探索和 replay sampling 分别使用显式随机源。全程 CPU 单线程。训练日志记录 episode return，但模型选择不读取最终测试种子。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from collections import deque  # 导入本单元所需的依赖。
from copy import deepcopy  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from hashlib import sha256  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 4101  # 计算并保存当前步骤的中间状态。
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。
assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "seed": SEED})  # 执行当前语句以推进本节示例。

## 2. 环境：可手算的 chain MDP

状态是 0–5 的位置，动作 0/1 表示左/右。到达位置 5 得 `+1` 并真正 terminated；其他步 `-0.02`。达到最大步数只是 truncated，Bellman target 是否 bootstrap 应由任务语义决定，不能把 terminated 与 time-limit truncation 混为一谈。

网络输入使用 one-hot，避免把位置编号错误解释为连续尺度。

In [ ]:
class ChainEnv:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, size=6, max_steps=12):  # 定义本节可复用的核心函数。
        if size < 3 or max_steps < size - 1:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_chain_config")  # 遇到非法合同立即显式失败。
        self.size, self.max_steps = size, max_steps  # 计算并保存当前步骤的中间状态。
        self.reset()  # 执行当前语句以推进本节示例。
    def observation(self):  # 定义本节可复用的核心函数。
        return F.one_hot(torch.tensor(self.position), self.size).float()  # 返回当前分支计算出的结果。
    def reset(self, start=0):  # 定义本节可复用的核心函数。
        if not 0 <= start < self.size - 1:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_start")  # 遇到非法合同立即显式失败。
        self.position, self.steps = int(start), 0  # 计算并保存当前步骤的中间状态。
        return self.observation()  # 返回当前分支计算出的结果。
    def step(self, action):  # 定义本节可复用的核心函数。
        if action not in (0, 1):  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_action")  # 遇到非法合同立即显式失败。
        self.position = max(0, self.position - 1) if action == 0 else min(self.size - 1, self.position + 1)  # 计算并保存当前步骤的中间状态。
        self.steps += 1  # 计算并保存当前步骤的中间状态。
        terminated = self.position == self.size - 1  # 计算并保存当前步骤的中间状态。
        truncated = self.steps >= self.max_steps and not terminated  # 计算并保存当前步骤的中间状态。
        reward = 1.0 if terminated else -0.02  # 计算并保存当前步骤的中间状态。
        return self.observation(), reward, terminated, truncated  # 返回当前分支计算出的结果。

env_probe = ChainEnv()  # 计算并保存当前步骤的中间状态。
state = env_probe.reset()  # 计算并保存当前步骤的中间状态。
for _ in range(5): state, reward, terminated, truncated = env_probe.step(1)  # 遍历输入元素以累积或检查结果。
assert terminated and not truncated and reward == 1.0  # 用受控断言验证关键不变量。
assert int(state.argmax()) == 5  # 用受控断言验证关键不变量。
assert ChainEnv(max_steps=5).max_steps == 5  # 用受控断言验证关键不变量。

## 3. QNetwork 与动作合同

`forward([B,state_dim]) -> [B,action_dim]` 输出每个动作的未归一化 Q 值，不做 softmax。Q 值是折扣回报估计，不是动作概率。服务时 argmax；训练探索只发生在 agent 边界。

In [ ]:
class QNetwork(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, state_dim, action_dim, hidden_dim=32):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.state_dim, self.action_dim, self.hidden_dim = state_dim, action_dim, hidden_dim  # 计算并保存当前步骤的中间状态。
        self.network = nn.Sequential(nn.Linear(state_dim, hidden_dim), nn.ReLU(),  # 计算并保存当前步骤的中间状态。
                                     nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),  # 执行当前语句以推进本节示例。
                                     nn.Linear(hidden_dim, action_dim))  # 执行当前语句以推进本节示例。
    def forward(self, states):  # 定义本节可复用的核心函数。
        if states.ndim != 2 or states.shape[1] != self.state_dim:  # 按当前条件选择后续控制路径。
            raise ValueError("expected_batch_state")  # 遇到非法合同立即显式失败。
        if not torch.isfinite(states).all():  # 按当前条件选择后续控制路径。
            raise ValueError("nonfinite_state")  # 遇到非法合同立即显式失败。
        return self.network(states)  # 返回当前分支计算出的结果。

q_probe = QNetwork(6, 2)  # 计算并保存当前步骤的中间状态。
assert q_probe(torch.eye(6)).shape == (6, 2)  # 用受控断言验证关键不变量。
assert not any(isinstance(layer, nn.Softmax) for layer in q_probe.modules())  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    q_probe(torch.zeros(6))  # 执行当前语句以推进本节示例。
    raise AssertionError("unbatched state must fail")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。

## 4. 经验回放：数据 schema 与随机采样

每条 transition 保存 `state, action, reward, next_state, terminated`。truncated 不写进 terminal mask，本例结束 episode 后仍允许对 time-limit transition bootstrap。生产 replay 还要处理容量、优先级、并发写、策略版本和敏感日志保留。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Transition:  # 定义承载本节状态与行为的数据结构。
    state: torch.Tensor  # 执行当前语句以推进本节示例。
    action: int  # 执行当前语句以推进本节示例。
    reward: float  # 执行当前语句以推进本节示例。
    next_state: torch.Tensor  # 执行当前语句以推进本节示例。
    terminated: bool  # 执行当前语句以推进本节示例。
    truncated: bool  # 执行当前语句以推进本节示例。

class ReplayBuffer:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, capacity, seed):  # 定义本节可复用的核心函数。
        if capacity <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("capacity_must_be_positive")  # 遇到非法合同立即显式失败。
        self.data = deque(maxlen=capacity)  # 计算并保存当前步骤的中间状态。
        self.rng = np.random.default_rng(seed)  # 计算并保存当前步骤的中间状态。
    def append(self, transition):  # 定义本节可复用的核心函数。
        if transition.action not in (0, 1) or not math.isfinite(transition.reward):  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_transition")  # 遇到非法合同立即显式失败。
        if not isinstance(transition.terminated, (bool, np.bool_)) or not isinstance(transition.truncated, (bool, np.bool_)):  # 按当前条件选择后续控制路径。
            raise TypeError("transition_flags_must_be_bool")  # 遇到非法合同立即显式失败。
        if transition.state.ndim != 1 or transition.next_state.shape != transition.state.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("transition_state_shape")  # 遇到非法合同立即显式失败。
        if not torch.isfinite(transition.state).all() or not torch.isfinite(transition.next_state).all():  # 按当前条件选择后续控制路径。
            raise ValueError("nonfinite_transition_state")  # 遇到非法合同立即显式失败。
        self.data.append(transition)  # 执行当前语句以推进本节示例。
    def sample(self, batch_size):  # 定义本节可复用的核心函数。
        if not 1 <= batch_size <= len(self.data):  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_batch_size")  # 遇到非法合同立即显式失败。
        indices = self.rng.choice(len(self.data), size=batch_size, replace=False)  # 计算并保存当前步骤的中间状态。
        rows = [self.data[int(i)] for i in indices]  # 计算并保存当前步骤的中间状态。
        return (torch.stack([x.state for x in rows]),  # 返回当前分支计算出的结果。
                torch.tensor([x.action for x in rows], dtype=torch.long),  # 计算并保存当前步骤的中间状态。
                torch.tensor([x.reward for x in rows], dtype=torch.float32),  # 计算并保存当前步骤的中间状态。
                torch.stack([x.next_state for x in rows]),  # 执行当前语句以推进本节示例。
                torch.tensor([x.terminated for x in rows], dtype=torch.bool),  # 计算并保存当前步骤的中间状态。
                torch.tensor([x.truncated for x in rows], dtype=torch.bool))  # 计算并保存当前步骤的中间状态。
    def __len__(self): return len(self.data)  # 定义本节可复用的核心函数。

buffer_probe = ReplayBuffer(3, 7)  # 计算并保存当前步骤的中间状态。
for i in range(4):  # 遍历输入元素以累积或检查结果。
    s = F.one_hot(torch.tensor(i % 3), 6).float()  # 计算并保存当前步骤的中间状态。
    buffer_probe.append(Transition(s, i % 2, float(i), s, False, False))  # 执行当前语句以推进本节示例。
assert len(buffer_probe) == 3  # 用受控断言验证关键不变量。
assert buffer_probe.sample(2)[0].shape == (2, 6)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    bad_state41 = torch.zeros(6); bad_state41[0] = float("nan")  # 计算并保存当前步骤的中间状态。
    buffer_probe.append(Transition(bad_state41, 0, 0.0, torch.zeros(6), False, False))  # 执行当前语句以推进本节示例。
    raise AssertionError("NaN replay state must fail")  # 遇到非法合同立即显式失败。
except ValueError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "nonfinite_transition_state"  # 用受控断言验证关键不变量。
assert len(buffer_probe) == 3  # 用受控断言验证关键不变量。

## 5. Double DQN target 的职责分离

Double DQN 用 online network 选择 `argmax_a Q_online(s',a)`，再由 target network 对该动作估值：

`y = r + gamma * (1-terminated) * Q_target(s', argmax Q_online)`。

target 必须在 `no_grad` 下计算。terminal 样本严格不 bootstrap；普通 DQN 的 `max Q_target` 与 Double DQN 可能选出不同结果。

In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def double_dqn_targets(online, target, next_states, rewards, terminated, gamma):  # 定义本节可复用的核心函数。
    if not 0 <= gamma <= 1 or rewards.shape != terminated.shape or rewards.ndim != 1:  # 按当前条件选择后续控制路径。
        raise ValueError("invalid_target_contract")  # 遇到非法合同立即显式失败。
    if terminated.dtype != torch.bool or not rewards.is_floating_point():  # 按当前条件选择后续控制路径。
        raise TypeError("target_flags_or_rewards_dtype")  # 遇到非法合同立即显式失败。
    if next_states.ndim != 2 or next_states.shape[0] != len(rewards):  # 按当前条件选择后续控制路径。
        raise ValueError("target_batch_mismatch")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(next_states).all() or not torch.isfinite(rewards).all():  # 按当前条件选择后续控制路径。
        raise ValueError("nonfinite_target_input")  # 遇到非法合同立即显式失败。
    next_actions = online(next_states).argmax(dim=1)  # 计算并保存当前步骤的中间状态。
    next_values = target(next_states).gather(1, next_actions[:, None]).squeeze(1)  # 计算并保存当前步骤的中间状态。
    return rewards + gamma * (~terminated).float() * next_values  # 返回当前分支计算出的结果。

class FixedQ(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, values):  # 定义本节可复用的核心函数。
        super().__init__(); self.register_buffer("values", torch.tensor(values, dtype=torch.float32))  # 计算并保存当前步骤的中间状态。
    def forward(self, states): return self.values[: len(states)]  # 定义本节可复用的核心函数。

online_fixed = FixedQ([[1,3], [5,2]])  # 计算并保存当前步骤的中间状态。
target_fixed = FixedQ([[10,7], [4,9]])  # 计算并保存当前步骤的中间状态。
oracle_targets = double_dqn_targets(online_fixed, target_fixed, torch.zeros(2,6),  # 计算并保存当前步骤的中间状态。
                                    torch.tensor([0.5,2.0]), torch.tensor([False,True]), 0.9)  # 执行当前语句以推进本节示例。
assert torch.allclose(oracle_targets, torch.tensor([6.8, 2.0]))  # 用受控断言验证关键不变量。
assert not oracle_targets.requires_grad  # 用受控断言验证关键不变量。
plain_dqn_first = 0.5 + 0.9 * 10.0  # 计算并保存当前步骤的中间状态。
assert not math.isclose(float(oracle_targets[0]), plain_dqn_first)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    double_dqn_targets(online_fixed, target_fixed, torch.zeros(2,6), torch.tensor([0.5,2.0]), torch.tensor([0,1]), 0.9)  # 执行当前语句以推进本节示例。
    raise AssertionError("integer terminated mask must fail")  # 遇到非法合同立即显式失败。
except TypeError as error:  # 捕获预期异常并验证失败分支。
    assert str(error) == "target_flags_or_rewards_dtype"  # 用受控断言验证关键不变量。
assert oracle_targets.shape == (2,)  # 用受控断言验证关键不变量。

## 6. 训练循环：探索、replay 与 target 同步

前若干 transition 先填充 replay。epsilon 逐步下降但保留探索下限；每次更新只对 chosen action 的 Q 计算 Huber loss。target network 定期硬同步且从不被 optimizer 更新。

下面固定 episode 数，不根据 test return 调参。真实项目应保存环境版本、reward 定义、行为策略和评估种子。

In [ ]:
def choose_action(network, state, epsilon, rng):  # 定义本节可复用的核心函数。
    if not 0 <= epsilon <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError("epsilon_out_of_range")  # 遇到非法合同立即显式失败。
    if rng.random() < epsilon:  # 按当前条件选择后续控制路径。
        return int(rng.integers(0, network.action_dim))  # 返回当前分支计算出的结果。
    with torch.no_grad(): return int(network(state[None]).argmax(dim=1).item())  # 在受管理的上下文中执行操作。

torch.manual_seed(SEED + 1)  # 执行当前语句以推进本节示例。
online41, target41 = QNetwork(6,2), QNetwork(6,2)  # 计算并保存当前步骤的中间状态。
target41.load_state_dict(online41.state_dict()); target41.requires_grad_(False); target41.eval()  # 执行当前语句以推进本节示例。
optimizer41 = torch.optim.Adam(online41.parameters(), lr=0.006)  # 计算并保存当前步骤的中间状态。
replay41 = ReplayBuffer(500, SEED + 2)  # 计算并保存当前步骤的中间状态。
env41 = ChainEnv()  # 计算并保存当前步骤的中间状态。
explore_rng = np.random.default_rng(SEED + 3)  # 计算并保存当前步骤的中间状态。
TARGET_SYNC_INTERVAL41 = 25  # 计算并保存当前步骤的中间状态。
BOOTSTRAP_ON_TRUNCATION41 = True  # 计算并保存当前步骤的中间状态。
returns41, losses41 = [], []  # 返回当前分支计算出的结果。
initial_online41 = {k: v.detach().clone() for k,v in online41.state_dict().items()}  # 计算并保存当前步骤的中间状态。
updates41 = 0  # 计算并保存当前步骤的中间状态。
for episode in range(120):  # 遍历输入元素以累积或检查结果。
    state = env41.reset(start=int(explore_rng.integers(0, 3)))  # 计算并保存当前步骤的中间状态。
    episode_return = 0.0  # 计算并保存当前步骤的中间状态。
    epsilon = max(0.05, 0.9 - episode / 100)  # 计算并保存当前步骤的中间状态。
    for _ in range(env41.max_steps):  # 遍历输入元素以累积或检查结果。
        action = choose_action(online41, state, epsilon, explore_rng)  # 计算并保存当前步骤的中间状态。
        next_state, reward, terminated, truncated = env41.step(action)  # 计算并保存当前步骤的中间状态。
        replay41.append(Transition(state.clone(), action, reward, next_state.clone(), terminated, truncated))  # 执行当前语句以推进本节示例。
        state, episode_return = next_state, episode_return + reward  # 计算并保存当前步骤的中间状态。
        if len(replay41) >= 32:  # 按当前条件选择后续控制路径。
            states, actions, rewards, next_states, terminals, truncations = replay41.sample(32)  # 计算并保存当前步骤的中间状态。
            assert truncations.dtype == torch.bool  # 用受控断言验证关键不变量。
            predicted = online41(states).gather(1, actions[:,None]).squeeze(1)  # 计算并保存当前步骤的中间状态。
            targets = double_dqn_targets(online41, target41, next_states, rewards, terminals, gamma=0.95)  # 计算并保存当前步骤的中间状态。
            loss = F.smooth_l1_loss(predicted, targets)  # 计算并保存当前步骤的中间状态。
            optimizer41.zero_grad(set_to_none=True); loss.backward()  # 计算并保存当前步骤的中间状态。
            torch.nn.utils.clip_grad_norm_(online41.parameters(), 5.0); optimizer41.step()  # 执行当前语句以推进本节示例。
            losses41.append(float(loss.detach())); updates41 += 1  # 计算并保存当前步骤的中间状态。
            if updates41 % TARGET_SYNC_INTERVAL41 == 0: target41.load_state_dict(online41.state_dict())  # 按当前条件选择后续控制路径。
        if terminated or truncated: break  # 按当前条件选择后续控制路径。
    returns41.append(episode_return)  # 返回当前分支计算出的结果。

target41.load_state_dict(online41.state_dict()); target41.eval(); online41.eval()  # 执行当前语句以推进本节示例。
changed41 = any(not torch.equal(initial_online41[k], v) for k,v in online41.state_dict().items())  # 计算并保存当前步骤的中间状态。
assert changed41 and updates41 > 100  # 用受控断言验证关键不变量。
assert np.mean(returns41[-30:]) > np.mean(returns41[:30])  # 用受控断言验证关键不变量。
assert losses41 and np.isfinite(losses41).all()  # 用受控断言验证关键不变量。
print({"mean_return_first30": np.mean(returns41[:30]), "last30": np.mean(returns41[-30:]), "updates": updates41})  # 执行当前语句以推进本节示例。

## 7. 独立 greedy 评估与 Q 值诊断

评估 epsilon 固定为 0，不写 replay、不更新权重。除了 success rate，还检查从每个非终止位置的 greedy action 和 Q 值有限性。链式环境的最优动作始终向右，是一个可解释 oracle。

In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def evaluate_policy(network, starts):  # 定义本节可复用的核心函数。
    successes, lengths = [], []  # 计算并保存当前步骤的中间状态。
    before = {k: v.clone() for k,v in network.state_dict().items()}  # 计算并保存当前步骤的中间状态。
    for start in starts:  # 遍历输入元素以累积或检查结果。
        env = ChainEnv(); state = env.reset(start)  # 计算并保存当前步骤的中间状态。
        for step in range(env.max_steps):  # 遍历输入元素以累积或检查结果。
            action = int(network(state[None]).argmax(1).item())  # 计算并保存当前步骤的中间状态。
            state, _, terminated, truncated = env.step(action)  # 计算并保存当前步骤的中间状态。
            if terminated or truncated: break  # 按当前条件选择后续控制路径。
        successes.append(terminated); lengths.append(step + 1)  # 执行当前语句以推进本节示例。
    assert all(torch.equal(before[k], v) for k,v in network.state_dict().items())  # 用受控断言验证关键不变量。
    return float(np.mean(successes)), lengths  # 返回当前分支计算出的结果。

success_rate41, eval_lengths41 = evaluate_policy(online41, [0,1,2,3,4] * 5)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    all_q41 = online41(torch.eye(6)[:5]); greedy_actions41 = all_q41.argmax(1)  # 计算并保存当前步骤的中间状态。
print({"success_rate": success_rate41, "greedy_actions": greedy_actions41.tolist(), "lengths": eval_lengths41[:5]})  # 执行当前语句以推进本节示例。
assert success_rate41 == 1.0  # 用受控断言验证关键不变量。
assert torch.equal(greedy_actions41, torch.ones(5, dtype=torch.long))  # 用受控断言验证关键不变量。
assert torch.isfinite(all_q41).all()  # 用受控断言验证关键不变量。

## 8. 策略制品、动作语义与在线边界

仅保存 state_dict 不够：动作 `0/1` 的语义、state encoder、环境/reward 版本、gamma 与网络 config 都必须绑定。公开接口只按受信 policy version 加载模型，并在每次决策前校验；客户端不能提交任意 `nn.Module`。

真实线上 RL 还需要离线安全评估、行为策略覆盖、约束动作过滤、熔断与人工接管。hash 只做一致性检查，发布仍需签名。

In [ ]:
def state_hash41(module):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for name,value in sorted(module.state_dict().items()):  # 遍历输入元素以累积或检查结果。
        array=value.detach().cpu().contiguous().numpy()  # 计算并保存当前步骤的中间状态。
        digest.update(name.encode()); digest.update(str(array.dtype).encode())  # 执行当前语句以推进本节示例。
        digest.update(json.dumps(list(array.shape)).encode()); digest.update(array.tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

def config41(model): return {"state_dim": model.state_dim, "action_dim": model.action_dim, "hidden_dim": model.hidden_dim}  # 定义本节可复用的核心函数。
def bundle_hash41(value):  # 定义本节可复用的核心函数。
    return sha256(json.dumps({k:v for k,v in value.items() if k!="bundle_sha256"},sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

artifact41 = {"policy_version":"double-dqn-chain-v1", "architecture":"QNetwork",  # 计算并保存当前步骤的中间状态。
              "config":config41(online41), "state_sha256":state_hash41(online41),  # 执行当前语句以推进本节示例。
              "state_encoder":"chain-one-hot-v1", "action_map":{"0":"left","1":"right"},  # 执行当前语句以推进本节示例。
              "environment_version":"chain-6-v1", "reward_version":"goal1-step-0.02-v1", "gamma":0.95,  # 执行当前语句以推进本节示例。
              "training_semantics":{"bootstrap_on_truncation":BOOTSTRAP_ON_TRUNCATION41,  # 执行当前语句以推进本节示例。
                                    "target_sync_interval":TARGET_SYNC_INTERVAL41,  # 执行当前语句以推进本节示例。
                                    "replay_capacity":500,"training_episodes":120,"seed":SEED}}  # 执行当前语句以推进本节示例。
artifact41["bundle_sha256"] = bundle_hash41(artifact41)  # 计算并保存当前步骤的中间状态。
_REGISTRY41 = {artifact41["policy_version"]:{"model":online41,"artifact":deepcopy(artifact41)}}  # 计算并保存当前步骤的中间状态。

def load_policy41(version):  # 定义本节可复用的核心函数。
    if version not in _REGISTRY41: raise KeyError("unknown_policy")  # 按当前条件选择后续控制路径。
    entry=_REGISTRY41[version]; model,artifact=entry["model"],entry["artifact"]  # 计算并保存当前步骤的中间状态。
    if type(model) is not QNetwork or artifact["architecture"] != type(model).__name__: raise TypeError("architecture_mismatch")  # 按当前条件选择后续控制路径。
    if artifact["bundle_sha256"] != bundle_hash41(artifact): raise RuntimeError("bundle_mismatch")  # 按当前条件选择后续控制路径。
    if artifact["config"] != config41(model) or artifact["state_sha256"] != state_hash41(model): raise RuntimeError("model_mismatch")  # 按当前条件选择后续控制路径。
    semantics = artifact.get("training_semantics", {})  # 计算并保存当前步骤的中间状态。
    if semantics.get("bootstrap_on_truncation") is not True or semantics.get("target_sync_interval") != TARGET_SYNC_INTERVAL41:  # 按当前条件选择后续控制路径。
        raise RuntimeError("training_semantics_mismatch")  # 遇到非法合同立即显式失败。
    return model,artifact  # 返回当前分支计算出的结果。

@torch.no_grad()  # 为下方定义附加声明式配置。
def policy_action41(state, version="double-dqn-chain-v1"):  # 定义本节可复用的核心函数。
    model,artifact=load_policy41(version)  # 计算并保存当前步骤的中间状态。
    raw=torch.as_tensor(state,dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
    if raw.shape != (artifact["config"]["state_dim"],) or not torch.isfinite(raw).all(): raise ValueError("invalid_online_state")  # 按当前条件选择后续控制路径。
    if not torch.allclose(raw.sum(),torch.tensor(1.0)) or not bool(((raw==0)|(raw==1)).all()): raise ValueError("state_encoder_mismatch")  # 按当前条件选择后续控制路径。
    action=int(model(raw[None]).argmax(1).item())  # 计算并保存当前步骤的中间状态。
    return action,{"policy_version":version,"action_name":artifact["action_map"][str(action)],"bundle_sha256":artifact["bundle_sha256"]}  # 返回当前分支计算出的结果。

served_action41, trace41 = policy_action41(torch.eye(6)[0])  # 计算并保存当前步骤的中间状态。
assert served_action41 == 1 and trace41["action_name"] == "right"  # 用受控断言验证关键不变量。
parameter41=next(online41.parameters()); backup41=parameter41.detach().clone()  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    with torch.no_grad(): parameter41.add_(0.25)  # 在受管理的上下文中执行操作。
    try: policy_action41(torch.eye(6)[0]); raise AssertionError("tampered policy must fail")  # 尝试执行可能失败的受控操作。
    except RuntimeError as error: assert str(error)=="model_mismatch"  # 捕获预期异常并验证失败分支。
finally:  # 无论结果如何都执行收尾逻辑。
    with torch.no_grad(): parameter41.copy_(backup41)  # 在受管理的上下文中执行操作。
action_map_backup41 = deepcopy(_REGISTRY41["double-dqn-chain-v1"]["artifact"]["action_map"])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    _REGISTRY41["double-dqn-chain-v1"]["artifact"]["action_map"]["1"] = "unsafe-remap"  # 计算并保存当前步骤的中间状态。
    try: load_policy41("double-dqn-chain-v1"); raise AssertionError("tampered action contract must fail")  # 尝试执行可能失败的受控操作。
    except RuntimeError as error: assert str(error)=="bundle_mismatch"  # 捕获预期异常并验证失败分支。
finally:  # 无论结果如何都执行收尾逻辑。
    _REGISTRY41["double-dqn-chain-v1"]["artifact"]["action_map"] = action_map_backup41  # 计算并保存当前步骤的中间状态。
try: policy_action41(torch.zeros(6)); raise AssertionError("invalid one-hot must fail")  # 尝试执行可能失败的受控操作。
except ValueError as error: assert str(error)=="state_encoder_mismatch"  # 捕获预期异常并验证失败分支。
assert served_action41 in (0, 1)  # 用受控断言验证关键不变量。

## 9. 面试总结与来源

DQN 的核心不是 MLP，而是用 replay 降低样本相关性、用 target network 稳定 bootstrap。Double DQN 把动作选择与估值拆开：`online(next_state).argmax` 负责选动作，`target(next_state)` 只对该动作估值，从而缓解同一组噪声同时参与选择和估值造成的过估计。

目标值是 `r + gamma * (1-terminated) * Q_target(s', argmax_a Q_online(s',a))`。时间上限导致的 `truncated` 是否继续 bootstrap 是任务语义，不能偷懒并入 terminal；本例把该选择连同 target 同步周期、replay 容量和随机种子写进发布合同。上线还要检查离线数据 coverage、行为策略偏差、动作约束、回滚阈值与 shadow/canary 指标。

- Mnih et al., *Human-level control through deep reinforcement learning*：https://www.nature.com/articles/nature14236
- van Hasselt et al., *Deep Reinforcement Learning with Double Q-learning*：https://arxiv.org/abs/1509.06461
- PyTorch autograd notes：https://pytorch.org/docs/stable/notes/autograd.html

这里没有 prioritized replay、n-step return、distributional Q、连续动作或真实离线策略评估。

In [ ]:
assert type(online41).__name__ == "QNetwork"  # 用受控断言验证关键不变量。
assert not any(p.requires_grad for p in target41.parameters())  # 用受控断言验证关键不变量。
assert optimizer41.param_groups[0]["params"][0] is next(online41.parameters())  # 用受控断言验证关键不变量。
assert success_rate41 == 1.0  # 用受控断言验证关键不变量。
assert all(action == 1 for action in greedy_actions41.tolist())  # 用受控断言验证关键不变量。
assert state_hash41(online41) == artifact41["state_sha256"]  # 用受控断言验证关键不变量。
assert bundle_hash41(artifact41) == artifact41["bundle_sha256"]  # 用受控断言验证关键不变量。
assert policy_action41(torch.eye(6)[4])[0] == 1  # 用受控断言验证关键不变量。
print("Double DQN、终止语义、评估与可信策略回归全部通过。")  # 执行当前语句以推进本节示例。